# 07 Model error

Does each design still image the BDT if the real velocity structure differs from the model? Echoes are made through
perturbed "true" models; focusing uses the best-estimate 1D and 3D models. Truth tables are on a coarse grid, so
variants are compared against a coarse-grid baseline. R is noise-free.

In [1]:
%run mi_common.py
import os, re
import skfmm

In [2]:
# --- Truth variants: how the real Earth might differ from the model ---
VARIANTS = {
    "baseline":                {},                    # unchanged model, coarse grid
    "all 5% faster":           dict(scale=1.05),      # whole model; no new tables needed
    "all 5% slower":           dict(scale=0.95),
    "shallow 10% faster":      dict(shallow=1.10),    # everything above the greywacke
    "shallow 10% slower":      dict(shallow=0.90),
    "clay-rich 50% thicker":   dict(clay=1.5),        # clay-rich base 50% deeper
    "greywacke 0.3 km higher": dict(gw=-0.3),         # greywacke top shallower everywhere
}

# --- Coarse grid for truth tables ---
CDXY, CDZ, CZMAX = 0.5, 0.2, 12.0      # km; coarse enough to be fast, fine enough for 2-6 Hz
VAR_DIR = "../data/tt_var"

# --- What to run ---
REFS_E = ["1d", "3d"]                  # focus with best 1D, or best 3D (fine tables from 03)
RUN = [n for n, m_ in S["scenarios"].items() if m_["shape"] == "dome"]
slug = lambda s: re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")
print(len(VARIANTS), "variants x", len(RUN), "scenarios")

7 variants x 15 scenarios


In [3]:
m = np.load("../data/velocity_model.npz")
x, y, z, vp0 = m["x"], m["y"], m["z"], m["vp"]
gw, cb, an = m["gw_top"][..., None], m["clay_base"][..., None], m["an_top"][..., None]
Z3 = z[None, None, :]
DZ = z[1] - z[0]


def column_value(vp, depth):
    """vp in each column at a given depth map (nearest cell)."""
    k = np.clip(np.round(depth / DZ).astype(int), 0, len(z) - 1)
    return np.take_along_axis(vp, k, axis=2)


def variant_model(v):
    """Best model with one change applied."""
    vp = vp0.copy()
    if "shallow" in v:
        vp = np.where(Z3 < gw, vp * v["shallow"], vp)
    if "clay" in v:
        new = np.minimum(cb * v["clay"], an)
        vp = np.where((Z3 >= cb) & (Z3 < new), column_value(vp0, np.full_like(cb, 0.2)), vp)
    if "gw" in v:
        new = gw + v["gw"]
        if v["gw"] < 0:      # raised: greywacke fills in
            vp = np.where((Z3 >= new) & (Z3 < gw), column_value(vp0, gw + DZ), vp)
        else:                # lowered: andesite fills in
            vp = np.where((Z3 >= gw) & (Z3 < new), column_value(vp0, gw - DZ), vp)
    return vp

In [ ]:
# --- Coarse grid ---
step_xy, step_z = int(round(CDXY / (x[1] - x[0]))), int(round(CDZ / DZ))
kz = z <= CZMAX
xc, yc, zc_ = x[::step_xy], y[::step_xy], z[kz][::step_z]
XXc, YYc, ZZc = np.meshgrid(xc, yc, zc_, indexing="ij")
ixc, iyc = np.abs(xc) <= 13, np.abs(yc) <= 13
R0 = 0.6                               # starting sphere, larger on the coarse grid (km)


def travel_time_coarse(sx, sy, v):
    """P travel time (s) from a station to every coarse cell."""
    d = np.sqrt((XXc - sx) ** 2 + (YYc - sy) ** 2 + ZZc ** 2)
    v0 = v[np.argmin(abs(xc - sx)), np.argmin(abs(yc - sy)), 0]
    t = skfmm.travel_time(d - R0, v, dx=[CDXY, CDXY, CDZ], order=2)
    return np.where(d < R0, d / v0, t + R0 / v0)[ixc][:, iyc].astype(np.float16)


mi_stations = stations.index[stations.type != "trillium"]
for name, v in VARIANTS.items():
    if "scale" in v:
        continue                       # uses the baseline tables, rescaled
    folder = f"{VAR_DIR}/{slug(name)}"
    os.makedirs(folder, exist_ok=True)
    vc = variant_model(v)[::step_xy, ::step_xy][:, :, kz][:, :, ::step_z]
    for s in mi_stations:
        path = f"{folder}/{s:03d}_p.npy"
        if not os.path.exists(path):
            np.save(path, travel_time_coarse(stations.x[s], stations.y[s], vc))
    print(name, "tables done")

np.savez(f"{VAR_DIR}/grid.npz", x=xc[ixc], y=yc[iyc], z=zc_)

baseline tables done
shallow 10% faster tables done
shallow 10% slower tables done
clay-rich 50% thicker tables done


In [ ]:
gcv = np.load(f"{VAR_DIR}/grid.npz")
gc0 = np.array([gcv["x"][0], gcv["y"][0], gcv["z"][0]])
dgc = np.array([CDXY, CDXY, CDZ])


def tt_true(st, pts, variant):
    """Travel times through a truth variant."""
    v = VARIANTS[variant]
    folder = f"{VAR_DIR}/{slug('baseline' if 'scale' in v else variant)}"
    idx = ((pts - gc0) / dgc).T
    T = np.array([map_coordinates(np.load(f"{folder}/{k:03d}_p.npy").astype(np.float32), idx, order=1)
                  for k in st])
    return T / v.get("scale", 1.0)


def reflection_true(st, scat, sigma, variant):
    """R_ij(f) with echoes travelling through a truth variant."""
    T, A = tt_true(st, scat, variant), 1 / (4 * np.pi * dist(st, scat))
    return np.array([(A * np.exp(-2j * np.pi * f * T) * sigma) @ (A * np.exp(-2j * np.pi * f * T)).T
                     for f in FREQS])

In [ ]:
rows = []
for variant in VARIANTS:
    for name in RUN:
        meta = S["scenarios"][name]
        c0, top = meta["centre"], sc[f"top_{name}"]
        near = (np.hypot(MX - c0[0], MY - c0[1]) <= SCAT_R) & ~np.isnan(top)
        scat = np.column_stack([MX[near], MY[near], top[near]])
        sigma = r0 * (1 + ROUGH * rng.normal(size=len(scat)))
        cx, cy, zc, pts = columns(c0)
        true_top = top_at(top, cx, cy)

        # phased: phase 1 decides with the best 1D model
        st1 = designs["phased"][0]["stations"]
        m1 = image(reflection_true(st1, scat, sigma, variant), st1, p1_pts, REF_DECIDE).reshape(V1x.shape).max(axis=2)
        k = np.unravel_index(m1.argmax(), m1.shape)
        chosen = (P1_H[k[0]], P1_H[k[1]])
        err = np.hypot(chosen[0] - c0[0], chosen[1] - c0[1])
        patches = phase2_patches(chosen)
        Rp = [reflection_true(p, scat, sigma, variant) for p in patches]
        for ref in REFS_E:
            I = sum(image(R, p, pts, ref) for R, p in zip(Rp, patches))
            rows.append((variant, name, "phased", ref, err, coverage(I, cx, cy, zc, true_top, c0)))

        # static designs
        for dname in STATIC:
            st_ = designs[dname][0]["stations"]
            R = reflection_true(st_, scat, sigma, variant)
            for ref in REFS_E:
                rows.append((variant, name, dname, ref, np.nan,
                             coverage(image(R, st_, pts, ref), cx, cy, zc, true_top, c0)))
    print(variant, "done")

err_res = pd.DataFrame(rows, columns=["variant", "scenario", "design", "ref", "p1_error", "coverage"])

In [ ]:
# --- Worst-case coverage for each variant and design ---
for ref in REFS_E:
    t = err_res[err_res.ref == ref].pivot_table(index="variant", columns="design", values="coverage", aggfunc="min")
    print(f"\nWorst-case coverage, focused with best {ref}")
    print(t.loc[list(VARIANTS)].round(2))

# --- Phase 1 location error for each variant ---
print("\nPhase 1 location error (km)")
p = err_res[(err_res.design == "phased") & (err_res.ref == REFS_E[0])]
print(p.groupby("variant").p1_error.agg(median="median", worst="max").loc[list(VARIANTS)].round(1))

In [ ]:
REF_SHOW = "1d"
t = (err_res[err_res.ref == REF_SHOW]
     .pivot_table(index="variant", columns="design", values="coverage", aggfunc="min")
     .loc[list(VARIANTS), ["phased"] + STATIC])

with plt.style.context("default"):
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    im = ax.imshow(t.values, cmap="viridis", vmin=0, vmax=1, aspect="auto")
    for i in range(t.shape[0]):
        for j in range(t.shape[1]):
            ax.text(j, i, f"{t.values[i, j]:.2f}", ha="center", va="center",
                    color="w" if t.values[i, j] < 0.5 else "k", fontsize=8)
    ax.set(xticks=range(t.shape[1]), xticklabels=t.columns, yticks=range(t.shape[0]), yticklabels=t.index,
           title=f"Worst-case coverage, focused with best {REF_SHOW}")
    fig.colorbar(im, label="Coverage of BDT top")
    plt.tight_layout()
    plt.show()